# <center>**Chapter 19 - Use PyTorch Deep Learning Models with scikit-learn**<center>

## **Mục tiêu**
- Bọc mạng `torch.nn.Module` bằng `skorch.NeuralNetClassifier` để sử dụng giao diện estimator của scikit-learn.
- Đánh giá mô hình với **Stratified K-Fold cross-validation** và báo cáo điểm từng fold cùng trung bình/độ lệch chuẩn.
- Thực hành trên bộ Sonar có sẵn tại `code/data/connectionist_bench_sonar_minesvsrocks/sonar.all-data`.

Bộ dữ liệu có 60 đặc trưng tín hiệu và nhãn `M` (mìn kim loại) hoặc `R` (đá). Phân tầng khi chia fold giúp giữ tỷ lệ lớp tương tự giữa các fold. Các fold huấn luyện được tạo lại độc lập, tránh dùng chung trọng số giữa các lần đánh giá.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from skorch import NeuralNetClassifier
from torch import nn, optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
def find_sonar_file() -> Path:
    # Tìm dữ liệu dù kernel chạy từ gốc dự án hay từ thư mục notebook.
    lab_dir_name = "Buổi thực hành số 7 (Optimizing DL Networks)"
    current = Path.cwd().resolve()
    for root in (current, *current.parents):
        candidates = (
            root / "data" / "connectionist_bench_sonar_minesvsrocks" / "sonar.all-data",
            root / "code" / "data" / "connectionist_bench_sonar_minesvsrocks" / "sonar.all-data",
            root / lab_dir_name / "code" / "data" / "connectionist_bench_sonar_minesvsrocks" / "sonar.all-data",
        )
        for candidate in candidates:
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Không tìm thấy tệp Sonar trong thư mục code/data.")

sonar = pd.read_csv(find_sonar_file(), header=None)
if sonar.shape[1] != 61:
    raise ValueError(f"Cần 60 đặc trưng và 1 nhãn; dữ liệu có {sonar.shape[1]} cột.")

In [3]:
X = sonar.iloc[:, :-1].to_numpy(dtype=np.float32)
encoder = LabelEncoder()
y = encoder.fit_transform(sonar.iloc[:, -1]).astype(np.int64)

print(f"Kích thước dữ liệu: {X.shape}")
print("Ánh xạ nhãn:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
print("Số mẫu mỗi lớp:", dict(zip(*np.unique(y, return_counts=True))))
print("Thiết bị:", DEVICE)

Kích thước dữ liệu: (208, 60)
Ánh xạ nhãn: {'M': np.int64(0), 'R': np.int64(1)}
Số mẫu mỗi lớp: {np.int64(0): np.int64(111), np.int64(1): np.int64(97)}
Thiết bị: cpu


## **Tạo mô hình PyTorch và bọc bằng skorch**

Mạng trả về hai logits, một cho mỗi lớp. `CrossEntropyLoss` trong skorch nhận logits và nhãn nguyên; không thêm softmax vào `forward`. `train_split=None` tắt validation nội bộ của skorch vì bước đánh giá được thực hiện bên ngoài bằng scikit-learn.

In [4]:
class SonarClassifier(nn.Module):
    def __init__(self, hidden_size: int = 60):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(60, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 2),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)

estimator = NeuralNetClassifier(
    module=SonarClassifier,
    module__hidden_size=60,
    criterion=nn.CrossEntropyLoss,
    optimizer=optim.Adam,
    lr=1e-3,
    max_epochs=40,
    batch_size=16,
    iterator_train__shuffle=True,
    train_split=None,
    device=DEVICE,
    verbose=0,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = cross_validate(
    estimator,
    X,
    y,
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=1,
    error_score="raise",
)

In [5]:
scores = pd.DataFrame(
    {
        "Fold": np.arange(1, len(results["test_score"]) + 1),
        "Accuracy train": results["train_score"],
        "Accuracy validation": results["test_score"],
        "Thời gian huấn luyện (giây)": results["fit_time"],
    }
)
display(scores.round(4))
print(
    "Accuracy validation: "
    f"{scores['Accuracy validation'].mean():.3f} "
    f"± {scores['Accuracy validation'].std(ddof=1):.3f}"
)

,Fold,Accuracy train,Accuracy validation,Thời gian huấn luyện (giây)
0,1,0.8614,0.7143,3.8719
1,2,0.8614,0.8571,0.6983
2,3,0.8554,0.6905,0.6157
3,4,0.8383,0.9024,0.6174
4,5,0.8383,0.8049,0.6640


Accuracy validation: 0.794 ± 0.091


## **Nhận xét**

Điểm trung bình ước lượng khả năng tổng quát hóa; độ lệch chuẩn cho biết kết quả phụ thuộc vào fold đến mức nào. Nếu accuracy train cao hơn nhiều validation, mô hình có thể đang quá khớp. Đây là ước lượng từ một cách chia có seed cố định, không phải cam kết hiệu năng trên mọi mẫu mới.

# <center>**HẾT**<center>